# ============================================================
# Calculate local elevation range using a 13x13 moving window
# ============================================================

In [1]:
import numpy as np
import rasterio
from scipy.ndimage import maximum_filter, minimum_filter

In [2]:
# ------------------------------------------------------------
# 1. Define input and output files
# ------------------------------------------------------------

input_dem = (
    "/home/jovyan/grupos/pgrad-disciplinas/"
    "CAP-423-ciencia-de-dados-geoespaciais/dataset/elevation.tif"
)

output_range = (
    "/home/jovyan/grupos/pgrad-disciplinas/"
    "CAP-423-ciencia-de-dados-geoespaciais/dataset/"
    "elevation_range_13x13.tif"
)

In [3]:
# ------------------------------------------------------------
# 2. Read the DEM
# ------------------------------------------------------------

with rasterio.open(input_dem) as src:

    dem = src.read(1).astype("float32")

    # Copy raster metadata so the output keeps the same
    # projection, extent, resolution, and geotransform.
    profile = src.profile.copy()

    nodata = src.nodata

/tmp/ipykernel_1163/3956679977.py:7: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  dem = src.read(1).astype("float32")


In [4]:
# ------------------------------------------------------------
# 3. Handle NoData values
# ------------------------------------------------------------

# Convert NoData values to NaN so they can be treated separately.
if nodata is not None:
    dem[dem == nodata] = np.nan

In [5]:
# ------------------------------------------------------------
# 4. Calculate local maximum and minimum elevation
# ------------------------------------------------------------

# A 13x13 window means:
#
#       6 pixels
#          ↑
#   ← 6 -- X -- 6 →
#          ↓
#       6 pixels
#
# For a raster with approximately 1 km spatial resolution,
# this represents a neighborhood extending about 6 km
# from the focal pixel in each direction.

window_size = 13

# maximum_filter and minimum_filter do not handle NaN values
# directly. Therefore, temporary arrays are created.
dem_for_max = np.where(np.isnan(dem), -np.inf, dem)
dem_for_min = np.where(np.isnan(dem),  np.inf, dem)

local_max = maximum_filter(
    dem_for_max,
    size=window_size,
    mode="nearest"
)

local_min = minimum_filter(
    dem_for_min,
    size=window_size,
    mode="nearest"
)


In [6]:
# ------------------------------------------------------------
# 5. Calculate local elevation range
# ------------------------------------------------------------

elevation_range = local_max - local_min

# Restore NoData where the original DEM is NoData.
elevation_range[np.isnan(dem)] = np.nan


In [7]:
# ------------------------------------------------------------
# 6. Save the result
# ------------------------------------------------------------

profile.update(
    dtype="float32",
    count=1,
    nodata=-9999,
    compress="deflate"
)

output_data = np.where(
    np.isnan(elevation_range),
    -9999,
    elevation_range
).astype("float32")

with rasterio.open(output_range, "w", **profile) as dst:
    dst.write(output_data, 1)

print("Finished.")
print(f"Output: {output_range}")

Finished.
Output: /home/jovyan/grupos/pgrad-disciplinas/CAP-423-ciencia-de-dados-geoespaciais/dataset/elevation_range_13x13.tif
